# HelixIR — GPU Benchmark (Colab)

Runs the three things that need real hardware and prints numbers to copy back:

1. **Serving benchmark** — real vLLM TTFT/TPOT vs. HelixIR's analytic prediction
2. **Attention kernels** — reference vs. Pallas vs. Triton flash-attention
3. **Inference sweep** — batch × context KV-cache / throughput / offload grid

**Before you run:** set the runtime to a GPU via *Runtime → Change runtime type → GPU*.
A free **T4** runs the analytic model + vLLM; **Pallas/Triton kernels need compute
capability ≥ 8.0 (A100/L4/H100)** and will be skipped on a T4.

> The KV-cache / serving / sweep / Triton code must be **on the branch you clone**.
> If it isn't pushed yet, commit + push it (or set `BRANCH` below), otherwise the
> new modules won't be there.

## 0 · Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv || echo 'NO GPU — set Runtime → GPU'

## 1 · Clone + install

Set `REPO` / `BRANCH` to wherever the inference module lives.

In [ ]:
REPO   = 'https://github.com/HackathonGroupMulti/HelixIR.git'
BRANCH = 'main'   # <- the branch that has helix/inference + attention_triton

import os
if not os.path.isdir('HelixIR'):
    !git clone --branch {BRANCH} --depth 1 {REPO}
%cd HelixIR
# Sanity check the new code is present:
assert os.path.isdir('helix/inference'), 'helix/inference missing — push it or fix BRANCH'
assert os.path.exists('helix/kernels/attention_triton.py'), 'attention_triton.py missing'
print('HelixIR checkout OK:', BRANCH)

In [ ]:
# JAX with CUDA + HelixIR itself. Colab already ships a torch+triton CUDA build.
!pip -q install -e . 
!pip -q install --upgrade 'jax[cuda12]'
# vLLM is a large install (a few minutes). Comment out to skip the serving section.
!pip -q install vllm
print('installs done')

In [ ]:
# Map the physical GPU to a HelixIR device string for the analytic roofline.
import torch

def helix_device():
    if not torch.cuda.is_available():
        return 'CPU'
    name = torch.cuda.get_device_name(0).upper()
    for key in ['H100', 'A100', 'L4', 'T4', 'V100', 'RTX 4090']:
        if key.replace(' ', '') in name.replace(' ', ''):
            return key
    return 'A100'

DEVICE = helix_device()
CC = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
PALLAS_OK = CC >= (8, 0)   # Pallas/Triton flash-attn need Ampere+
print('HelixIR device =', DEVICE, '| compute capability =', CC, '| Pallas/Triton ok =', PALLAS_OK)

## 2 · Analytic baseline (always runs)

These are HelixIR **predictions** — the reference the measured numbers get compared against.

In [ ]:
from helix.inference import ModelConfig, analyze_inference, print_inference_report

cfg = ModelConfig.from_preset('llama3-8b')
report = analyze_inference(cfg, batch=16, prompt_len=2048, gen_len=128, device=DEVICE)
print_inference_report(report)

## 3 · Serving benchmark — real vLLM vs. analytic

Uses **TinyLlama-1.1B** (ungated, small). The `ModelConfig` below matches TinyLlama's
shape so the analytic-vs-measured comparison is apples-to-apples.

In [ ]:
from helix.inference import serving_benchmark, print_serving_result

# TinyLlama-1.1B-Chat shape (L=22, 32 q-heads, 4 kv-heads GQA, head_dim=64).
tinyllama = ModelConfig(name='tinyllama-1.1b', num_layers=22, num_heads=32,
                        num_kv_heads=4, head_dim=64, ffn_dim=5632, vocab_size=32000)

res = serving_benchmark(
    tinyllama, batch=8, prompt_len=512, gen_len=64, device=DEVICE,
    model='TinyLlama/TinyLlama-1.1B-Chat-v1.0', backend='auto',
)
print_serving_result(res)
print('\nRAW:', res.to_dict())

## 4 · Attention kernels — reference vs. Pallas vs. Triton

Numerical correctness first, then a speed comparison. Skipped automatically on a T4
(compute capability < 8.0).

In [ ]:
# --- Numerical correctness: Triton flash-attn vs a reference ---
import math, torch
from helix.kernels.attention_triton import flash_attention_triton, triton_available

if triton_available() and PALLAS_OK:
    torch.manual_seed(0)
    B, S, H, D = 2, 256, 8, 64
    q, k, v = (torch.randn(B, S, H, D, device='cuda', dtype=torch.float16) for _ in range(3))
    def ref(q, k, v, causal):
        s = torch.einsum('bqhd,bkhd->bhqk', q.float(), k.float()) / math.sqrt(D)
        if causal:
            m = torch.tril(torch.ones(S, S, device='cuda', dtype=torch.bool))
            s = s.masked_fill(~m, float('-inf'))
        return torch.einsum('bhqk,bkhd->bqhd', torch.softmax(s, -1), v.float())
    for causal in (False, True):
        out = flash_attention_triton(q, k, v, causal=causal).float()
        err = (out - ref(q, k, v, causal)).abs().max().item()
        print(f'Triton causal={causal}: max abs err = {err:.4e}  ->  {"PASS" if err < 2e-2 else "FAIL"}')
else:
    print('Skipped: needs Triton + compute capability >= 8.0 (A100/L4/H100).')

In [ ]:
# --- Speed: reference (JAX) vs Pallas vs Triton ---
!helix compare attention --seq 512 --batch 8 --size 1024

## 5 · Inference sweep (batch × context)

Where does the KV cache overflow HBM and decode throughput collapse on *this* GPU?

In [ ]:
!helix infer-sweep llama3-8b --batches 1,8,32,64,128,256 --prompts 4096 --device {DEVICE}

## 6 · Copy back to Claude

Paste the outputs of sections **3**, **4**, and **5** back into the chat. In particular:

- Serving: measured TTFT / TPOT / throughput **and** the `% off` vs. analytic
- Kernels: the PASS/FAIL correctness line + the speed comparison table
- Sweep: the full table, noting the batch where `HBM` flips to `no`

Then we update HelixIR's README + résumé bullets with **measured** numbers.